In [1]:
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path
sys.path.append('..')
from src.preprocessing import scale_data, create_sequences, prepare_tensors
import pickle
import torch

# Dizinleri oluştur
Path('data/processed').mkdir(parents=True, exist_ok=True)

In [2]:
# Veri yükleme
df = pd.read_csv('data/raw/AMZN_2015-2025.csv', index_col=0, parse_dates=True)
print(f'Yüklenen veri: {len(df)} satır')
print(f'Kapanış fiyatı aralığı: {df["Close"].min():.2f} - {df["Close"].max():.2f}')

Yüklenen veri: 2516 satır
Kapanış fiyatı aralığı: 14.35 - 232.93


In [3]:
# Veri ölçekleme
scaled, scaler = scale_data(df, column='Close', train_ratio=0.8)
print(f'Scaled veri aralığı: {scaled.min():.4f} - {scaled.max():.4f}')
print(f'Scaler data_min: {scaler.data_min_}')
print(f'Scaler data_max: {scaler.data_max_}')

Scaled veri aralığı: -1.0000 - 1.5384
Scaler data_min: [14.34749985]
Scaler data_max: [186.57049561]


In [4]:
# Sequence oluşturma
X, y = create_sequences(scaled, lookback=20)
print(f'X shape: {X.shape}')
print(f'y shape: {y.shape}')

X shape: (2496, 20)
y shape: (2496,)


In [5]:
# Tensor hazırlama
(X_train, y_train), (X_test, y_test) = prepare_tensors(X, y, train_ratio=0.8)
print('Tensorler CPU üzerinde hazırlandı.')

📊 Tensorler Hazırlandı:
   Train set: X=torch.Size([1996, 20, 1]), y=torch.Size([1996, 1])
   Test set:  X=torch.Size([500, 20, 1]), y=torch.Size([500, 1])
Tensorler CPU üzerinde hazırlandı.


In [6]:
# Kaydetme işlemleri
scaler_path = 'data/processed/scaler.pkl'
tensor_path = 'data/processed/amzn_processed.pt'

# Scaler'ı pickle ile kaydet
with open(scaler_path, 'wb') as f:
    pickle.dump(scaler, f)
print(f'💾 Scaler kaydedildi: {scaler_path}')

# Tensorleri PyTorch ile kaydet
torch.save({
    'X_train': X_train,
    'y_train': y_train,
    'X_test': X_test,
    'y_test': y_test
}, tensor_path)
print(f'💾 Tensorler kaydedildi: {tensor_path}')

💾 Scaler kaydedildi: data/processed/scaler.pkl
💾 Tensorler kaydedildi: data/processed/amzn_processed.pt
